# Goal

Берём все запчасти из `18d_world_model_09:224`, а здесь только треним энкодер со встроенным `BatchNorm2d` (`is_ob_features_batch_norm=True`).

**Это перезапуск `32.1` с исправленным багом `WorldModel`, когда этот долбанный флаг не прокидывался `is_ob_features_batch_norm=True`!!!**

# TARGET_NOTEBOOK_FNAME

In [1]:
TARGET_NOTEBOOK_FNAME = '18d_world_model_10.ipynb'

# OPTUNA_DECISIVE_METRIC

In [2]:
OPTUNA_DECISIVE_METRIC = 'ssim'

# GRID_SEARCH_SPACE

In [3]:
# @launchit.collect
# GRID_SEARCH_SPACE = dict(
# )

# set_hyperparameters

In [4]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    import random
    HP.launch_goal = 'train_encoder'
    
    HP.general.comment = None
    HP.general.random_seed = random.randint(0, 100)
    HP.general.is_torch_deterministic = True
    HP.general.is_torch_compile = True
    HP.general.is_torch_amp = True

    HP.dataset.train = [
        '18b_dataset_02:8:cls=train,no=1,seq=8,mp=0,sh=3,210,160',
        '18b_dataset_02:8:cls=train,no=2,seq=8,mp=0,sh=3,210,160',
        '18b_dataset_02:8:cls=train,no=3,seq=8,mp=0,sh=3,210,160',
        '18b_dataset_02:8:cls=train,no=4,seq=8,mp=0,sh=3,210,160',
        '18b_dataset_02:8:cls=train,no=5,seq=8,mp=0,sh=3,210,160',
        '18b_dataset_02:8:cls=train,no=6,seq=8,mp=0,sh=3,210,160',
        '18b_dataset_02:8:cls=train,no=7,seq=8,mp=0,sh=3,210,160',
        '18b_dataset_02:8:cls=train,no=8,seq=8,mp=0,sh=3,210,160',
        '18b_dataset_02:8:cls=train,no=9,seq=8,mp=0,sh=3,210,160',
        '18b_dataset_02:8:cls=train,no=10,seq=8,mp=0,sh=3,210,160',
        '18b_dataset_02:8:cls=train,no=11,seq=8,mp=0,sh=3,210,160',
        '18b_dataset_02:8:cls=train,no=12,seq=8,mp=0,sh=3,210,160',
        
        '18b_dataset_04:1:cls=train,no=1,seq=8,mp=0,sh=3,210,160',
        '18b_dataset_04:1:cls=train,no=2,seq=8,mp=0,sh=3,210,160',
        '18b_dataset_04:1:cls=train,no=3,seq=8,mp=0,sh=3,210,160',
        '18b_dataset_04:1:cls=train,no=4,seq=8,mp=0,sh=3,210,160',
        '18b_dataset_04:1:cls=train,no=5,seq=8,mp=0,sh=3,210,160',
        '18b_dataset_04:1:cls=train,no=6,seq=8,mp=0,sh=3,210,160',
        '18b_dataset_04:1:cls=train,no=7,seq=8,mp=0,sh=3,210,160',
        '18b_dataset_04:1:cls=train,no=8,seq=8,mp=0,sh=3,210,160',
        '18b_dataset_04:1:cls=train,no=9,seq=8,mp=0,sh=3,210,160',
        '18b_dataset_04:1:cls=train,no=10,seq=8,mp=0,sh=3,210,160',
        '18b_dataset_04:1:cls=train,no=11,seq=8,mp=0,sh=3,210,160',
        '18b_dataset_04:1:cls=train,no=12,seq=8,mp=0,sh=3,210,160',
        '18b_dataset_04:1:cls=train,no=13,seq=8,mp=0,sh=3,210,160',
        '18b_dataset_04:1:cls=train,no=14,seq=8,mp=0,sh=3,210,160',
        '18b_dataset_04:1:cls=train,no=15,seq=8,mp=0,sh=3,210,160',
        '18b_dataset_04:1:cls=train,no=16,seq=8,mp=0,sh=3,210,160',
    ]
    HP.dataset.test = '18b_dataset_04:1:cls=test,no=1,seq=8,mp=0,sh=3,210,160'

    HP.model.parent = '18d_world_model_09:224'
    HP.model.input_sequence_length = 4
    HP.model.actions_count = 6
    HP.model.d_model = 256
    HP.model.transformer = dict(layers_count=3, heads_count=4, attention_backend=['EFFICIENT_ATTENTION', 'MATH'])
    HP.model.vision_head = dict(
        is_trainable=False, # ЗАМОРОЖЕННЫЙ
        ob_shape=(3, 89, 76),
        grid=(7, 6),
        features_counts=(32, 64, 128),
        normalization='pixel', 
    )
    HP.model.encoder = dict(
        is_trainable=True,
        is_ob_features_batch_norm=True,
        is_ob_tokens_layer_norm=True,
        parent=None, # ТРЕНИМ С НУЛЯ!
    )
    HP.model.predictor = dict(
        is_trainable=HP.launch_goal in ('train_predictor', 'train_all', None),
        pred_latents_source='separate',
        parent=None, # do not load from HP.model.parent
    )
    HP.model.renderer = dict(
        is_trainable=False, # ЗАМОРОЖЕННЫЙ
        ob_shape=(1, 178, 152),
        projector=dict(type='linear'), 
        render_engine=dict(
            type='layer', 
            layers_count=4, 
            features_counts=(96, 64, 32, 16),
        ),
    )
    HP.model.struct_projector = dict(
        is_trainable=HP.launch_goal in ('train_predictor', 'train_all', None),
        type='Identity',
        parent=None, # do not load from HP.model.parent
    )
    
    HP.train.epochs_count = 200
    HP.train.batch_size = 64
    HP.train.optimizer = 'AdamW'
    HP.train.max_grad_norm = 1.0
    HP.train.learn_rate = 'const(0.0005)'
    HP.train.bce_loss_coef = 'const(1.0)'
    HP.train.edge_loss_coef = 'const(1.0)'
    HP.train.recon_loss_coef = 'const(1.0)'
    HP.train.pred_loss = dict(
        type='MSE', 
        loss_coef='const(0.0)', 
    )
    
    return HP

# Results

Результаты норм (хотя немного ниже, чем `32.1`):
- `18d_world_model_10:20`, `ssim=0.9914112687110901`
- `18d_world_model_10:23`, `ssim=0.9911850690841675`
- `18d_world_model_10:21`, `ssim=0.9909895062446594`

**Выводы**
1) делаем второй шаг, чтобы заполировать

# System

In [5]:
import os, sys, re, subprocess, json
import IPython 
import concurrent.futures as cf
from collections import namedtuple

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
from optuna.trial import TrialState

project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]

sys.path.append(os.path.join(project_root_path, 'lib'))

from logging_utils import *
from math_utils import *
from artifact_registry import *
import launchit
import launch_dispatcher
from autoincrement import Autoincrement

In [6]:
CONFIG = namedtuple('CONFIG', 
                    'project_root_uri, model_group_uri, project_root_path, subproject_name, subproject_path, run_path, ' + 
                    'target_notebook_fname, target_notebook_name, ' + 
                    'optuna_study_notebook_fname, optuna_study_name, optuna_study_serial, optuna_study_fname')(
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    model_group_uri=None,
    project_root_path=project_root_path,
    subproject_name=None,
    subproject_path=os.path.abspath('../..'),
    run_path=None,
    target_notebook_fname=os.path.join(os.path.abspath('../..'), TARGET_NOTEBOOK_FNAME),
    target_notebook_name=None,
    optuna_study_notebook_fname=None,
    optuna_study_name=None,
    optuna_study_serial=None,
    optuna_study_fname=None,
)

with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as connection_file:
    optuna_study_notebook_fname = json.load(connection_file).get('jupyter_session')
    optuna_study_name, _ = os.path.splitext(os.path.basename(optuna_study_notebook_fname))
    optuna_study_serial = re.match(r'\w+_([\d\.\w]+)', optuna_study_name).group(1)
    optuna_study_fname = os.path.join(os.path.dirname(optuna_study_notebook_fname), optuna_study_name + '.optuna')
    CONFIG = CONFIG._replace(optuna_study_notebook_fname=optuna_study_notebook_fname)
    CONFIG = CONFIG._replace(optuna_study_name=optuna_study_name)
    CONFIG = CONFIG._replace(optuna_study_serial=optuna_study_serial)
    CONFIG = CONFIG._replace(optuna_study_fname=optuna_study_fname)

target_notebook_name, _ = os.path.splitext(os.path.basename(TARGET_NOTEBOOK_FNAME))
CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.target_notebook_fname)))
CONFIG = CONFIG._replace(model_group_uri=f'{CONFIG.project_root_uri}.{CONFIG.subproject_name}')
CONFIG = CONFIG._replace(target_notebook_name=target_notebook_name)
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
CONFIG._asdict()

{'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.18_rl',
 'project_root_path': '/home/misha/dev/mine/neurolab',
 'subproject_name': '18_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/18_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/18_rl',
 'target_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/18d_world_model_10.ipynb',
 'target_notebook_name': '18d_world_model_10',
 'optuna_study_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18d_study_33.1/18d_study_33.1.ipynb',
 'optuna_study_name': '18d_study_33.1',
 'optuna_study_serial': '33.1',
 'optuna_study_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18d_study_33.1/18d_study_33.1.optuna'}

In [7]:
LOG = Logging.get()
LOG.enable('syslog', False)
LOG.enable('stdout', False)
LOG.enable('verbose_stdout', True)

In [8]:
ARTIFACT_REGISTRY = ArtifactRegistry(maven_group_id=CONFIG.model_group_uri)

In [9]:
def create_optuna_launch():
    model_version = int(Autoincrement.get(f'{CONFIG.model_group_uri}.{CONFIG.target_notebook_name}'))
    assert model_version > 0, model_version
    ARTIFACT_REGISTRY.register_component(CONFIG.target_notebook_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    # Prep docker launch
    expandvars = dict(
        PROJECT_ROOT_PATH='/neurolab',
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.target_notebook_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL='TRAIN',
        OPTUNA_STUDY_FNAME=CONFIG.optuna_study_fname,
        OPTUNA_STUDY_NAME=CONFIG.optuna_study_name,
        OPTUNA_DECISIVE_METRIC=OPTUNA_DECISIVE_METRIC,
    )
    launch_fname = launchit.launchit(
        CONFIG.target_notebook_fname, 
        launch_serial=int(model_version),
        expandvars=expandvars, 
        make_py_file=False, 
        dir_name=CONFIG.run_path,
        collect_inds=['temp_config', 'optuna', 'initrd', 'build_docker_launch', 'optuna_run_docker_launch'],
        disable_inds=[],
    )
    return f'{CONFIG.target_notebook_name}:{model_version}', launch_fname

In [10]:
# Executed in a separate thread with GIL locked
def run_optuna_launch(launch_fname):
    # Run launch notebook locally, the latter will:
    # 1) sample values of hyperparameters from optuna study
    # 2) pack everything to docker launch (self-contained thing)
    # 3) run "docker_launch_run" cell which in turn will dispatch launch to cloud via launch_dispatcher
    # 4) collect result of a docker launch from cloud and update optuna study
    LOG(f'Launching "{launch_fname}"')
    
    subprocess.run(
        ['papermill', launch_fname, launch_fname, '--no-progress-bar'],
        capture_output=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )

    if os.path.exists(launch_fname + '.out'):
        with open(launch_fname + '.out', 'rt') as f:
            LOG(f.read())
    else:
        LOG(f'"{launch_fname}" completed with no output, probably failed')

## Unleash!

In [ ]:
optuna_study = optuna.create_study(
    study_name=CONFIG.optuna_study_name,
    directions=['maximize'],
    storage=JournalStorage(JournalFileBackend(file_path=CONFIG.optuna_study_fname)),
    load_if_exists=True,
)
optuna_study.set_user_attr('STUDY_SERIAL', CONFIG.optuna_study_serial)
launches_count = 6
completed_launches_count = 0

with LOG.auto_log_level(logging.INFO):
    with cf.ThreadPoolExecutor(max_workers=32) as executor:
        futures = {}
        idle_runners_af = RecursiveMovingAverageFilter(max_n=6)
        is_first_time = True
        
        while launches_count is None or completed_launches_count < launches_count:
            runners_info = launch_dispatcher.RunnersInfo.get()
            idle_runners_af(runners_info['idle'])

            if is_first_time or (idle_runners_af.n >= idle_runners_af.max_n and idle_runners_af.v >= 1):
                if launches_count is None or (completed_launches_count + len(futures) < launches_count):
                    launch_name, launch_fname = create_optuna_launch()
                    futures.update({executor.submit(run_optuna_launch, launch_fname): launch_name})
                    LOG(f'{idle_runners_af.v:.1f} idle runners exist, submitted launch "{launch_name}"; running launches={len(futures)}')
                    idle_runners_af.reset()
                    
                is_first_time = False

            try:
                while futures:
                    completed_futures, _ = cf.wait(futures, timeout=0.1, return_when=cf.FIRST_COMPLETED)

                    if not completed_futures:
                        break
                        
                    for completed_future in completed_futures:
                        launch_name = futures[completed_future]
                        del futures[completed_future]

                        exc = completed_future.exception()
                        
                        if exc is not None:
                            LOG(f'Launch "{launch_name}" failed: {exc}')
                        else:
                            LOG(f'Launch "{launch_name}" completed')
    
                    if completed_futures:
                        completed_launches_count += len(completed_futures)
                        LOG(f'{completed_launches_count} (+{len(completed_futures)}) launches completed; running launches={len(futures)}')
            except TimeoutError as e:
                pass

            time.sleep(5)

[I 2026-10-24 16:40:42,002] A new study created in Journal with name: 18d_study_33.1


2026.10.24-16:40:42.781139     0.752 >> Model instance registered, version=19
2026.10.24-16:40:42.838557     0.020 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18d_world_model_10-launch19.ipynb"
2026.10.24-16:40:42.839140     0.021 >> 8.0 idle runners exist, submitted launch "18d_world_model_10:19"; running launches=1
2026.10.24-16:41:14.085329     0.580 >> Model instance registered, version=20
2026.10.24-16:41:14.137669     0.017 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18d_world_model_10-launch20.ipynb"
2026.10.24-16:41:14.138826     0.001 >> 8.0 idle runners exist, submitted launch "18d_world_model_10:20"; running launches=2


In [15]:
study = optuna.create_study(
    study_name=optuna_study_name,
    storage=JournalStorage(JournalFileBackend(file_path=optuna_study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs.get('MODEL_VERSION', 'n/a')}')
    
    LOG('\tParams: ')
    
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    LOG(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        LOG(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        LOG(f"\tnumber: {trial.number}")
        LOG(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        LOG(f"\tparams: {trial.params}")
        LOG(f"\tvalues: {trial.values}")

[I 2026-10-24 17:34:32,748] Using an existing study with name '18d_study_33.1' instead of creating a new one.


2026.10.24-17:34:32.749550    64.606 >> Study statistics: 
2026.10.24-17:34:32.750167     0.001 >> 	Number of finished trials: 6
2026.10.24-17:34:32.750458     0.000 >> 	Number of pruned trials: 0
2026.10.24-17:34:32.750720     0.000 >> 	Number of complete trials: 5
2026.10.24-17:34:32.751180     0.000 >> Best trial:
2026.10.24-17:34:32.751675     0.000 >> 	Value: 0.9914112687110901
2026.10.24-17:34:32.751871     0.000 >> 	Model version: 20
2026.10.24-17:34:32.752122     0.000 >> 	Params: 


In [16]:
top_trials = sorted(filter(lambda t: t.values, study.trials), key=lambda t: -t.values[0])[:3]
LOG(f'Top trials:')

for i, trial in enumerate(top_trials):
    LOG(f'{i+1}) {CONFIG.target_notebook_name + ':' + trial.user_attrs['MODEL_VERSION']}, value={trial.values[0]}')

2026.10.24-17:34:33.563952     0.812 >> Top trials:
2026.10.24-17:34:33.565608     0.002 >> 1) 18d_world_model_10:20, value=0.9914112687110901
2026.10.24-17:34:33.566012     0.000 >> 2) 18d_world_model_10:23, value=0.9911850690841675
2026.10.24-17:34:33.566331     0.000 >> 3) 18d_world_model_10:21, value=0.9909895062446594


In [ ]:
values = []

for trial in study.trials:
    if trial.values is not None:
        values.append(trial.values[0])

values = np.array(values)
values.mean(), values.std()